# Self-RAG: Practical Demo (real model, 4-bit, Colab GPU)

In this notebook we run the real **Self-RAG model** (`selfrag/selfrag_llama2_7b`) in 4-bit on a free Colab T4 GPU, and use its **reflection tokens** to build a small Self-RAG pipeline.

**Part 1: Check that the model works (Sections 1 to 6)**
1. The free **T4 GPU** is available.
2. The real Self-RAG model loads in 4-bit.
3. The model really writes **reflection tokens** such as `[No Retrieval]` and `[Utility:5]`.

**Part 2: Build the Self-RAG pipeline (Sections 7 to 11)**
A small retriever, then four decisions read from the model's reflection tokens: retrieve or not, is the chunk relevant, is the answer supported, is it useful. We run it on five questions.

**Before you start:** Runtime → Change runtime type → **T4 GPU**, then Connect.

## 1. Check the GPU

In [ ]:
import torch

print('GPU available:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU: change the runtime type first')

GPU available: True
Tesla T4


## 2. Install the libraries

In [ ]:
pip install -U transformers accelerate bitsandbytes -q # q for silent installation

## 3. Load the real Self-RAG model in 4-bit

The model is about 13 GB, so it downloads to Colab's temporary disk (a few minutes). The 4-bit setting shrinks it so that it fits on the free T4 GPU. No token is needed, because the model is public.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = 'selfrag/selfrag_llama2_7b'

# Shrink the 7B model to 4-bit so it fits on the free T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = 'nf4',
    bnb_4bit_compute_dtype = torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map='auto')

## 4. Check that the reflection tokens exist in the vocabulary

If a token id equals the **unknown token id**, the model does not know that token as one piece.

In [ ]:
print('unknown token id:', tokenizer.unk_token_id)

for token in ['[No Retrieval]', '[Retrieval]', '[Relevant]', '[Fully supported]', '[Utility:5]']:
    print(token, '->', tokenizer.convert_tokens_to_ids(token))

unknown token id: 0
[No Retrieval] -> 32000
[Retrieval] -> 32001
[Relevant] -> 32004
[Fully supported] -> 32012
[Utility:5] -> 32011


## 5. Generate with the model card's own example

Self-RAG needs its **raw prompt format** (no chat template):
```
### Instruction:
{question}

### Response:
```
We keep the special tokens in the output, otherwise the reflection tokens would be hidden.

In [ ]:
# Raw Self-RAG prompt format from the model card
prompt = "### Instruction:\nLeave odd one out: twitter, instagram, whatsapp.\n\n### Response:\n"

inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=60, do_sample=False)  # do_sample=False: same answer every time

# Print only the new text, and keep the reflection tokens in it
print(tokenizer.decode(output_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=False))

[transformers] Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Twitter, Instagram, and WhatsApp are all popular social media platforms.[No Retrieval]However, WhatsApp is the odd one out as it is primarily a messaging app, while Twitter and Instagram are both primarily social media platforms.[Utility:5]</s>


## 6. How to read the result

- **Good:** Section 4 prints ids different from the unknown id, and Section 5's answer contains bracketed tokens like `[No Retrieval]` and `[Utility:5]`. The real Self-RAG model works, and Part 2 builds the full pipeline on top of it.
- **Bad:** Section 5 prints a normal answer with no bracketed tokens. Then the model is not producing reflection tokens. Check that you used the raw prompt format from Section 5 (no chat template) and `skip_special_tokens=False`.
- **If Colab crashes with "used all available RAM" in Section 3:** the model files (about 13 GB) did not fit in the free RAM. Try Kaggle instead, which has more RAM.

# Part 2: Build the Self-RAG pipeline

## 7. Set up a small retriever

Self-RAG needs a text to retrieve from. We use a short text about AI in healthcare, one chunk per sentence, stored with dense embeddings in memory.

### 7.1 Install the embedding library

In [ ]:
!pip install langchain-huggingface -q

### 7.2 The text, the chunks and the vector store

In [ ]:
import re
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

text = """Artificial Intelligence (AI) is transforming healthcare by improving disease diagnosis, patient monitoring, and treatment planning. Machine learning models analyze medical records, laboratory reports, and medical images to identify patterns and predict potential health risks.

Deep learning techniques, such as Convolutional Neural Networks (CNNs), are widely used for medical image classification. Natural Language Processing (NLP) helps extract meaningful information from clinical notes and electronic health records.

However, AI adoption in healthcare faces challenges, including data privacy, model bias, high implementation costs, and the need for reliable predictions. Combining AI technologies with human expertise can improve healthcare efficiency and patient outcomes."""

# One chunk per sentence
chunks = re.split(r'(?<=[.!?])\s+', text)

# Dense retriever: embeddings stored in memory
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
store = InMemoryVectorStore.from_texts(chunks, embedding=embeddings)

## 8. Helper functions for the four decisions

Self-RAG makes four decisions, each read from a reflection token:
1. **Retrieve or not?**
2. **Is the retrieved chunk relevant?**
3. **Is the answer supported by the chunk?**
4. **Is the answer useful?** (1 to 5)

### 8.1 Prompt format and token ids

The retrieved chunk goes into the prompt between `<paragraph>` tags, right after `[Retrieval]`.

In [ ]:
# Raw Self-RAG prompt format from the model card
def format_prompt(question, paragraph=None):
    prompt = f'### Instruction:\n{question}\n\n### Response:\n'
    if paragraph is not None:
        prompt += f'[Retrieval]<paragraph>{paragraph}</paragraph>'   # retrieved chunk goes here
    return prompt

retrieval_id = tokenizer.convert_tokens_to_ids('[Retrieval]')
no_retrieval_id = tokenizer.convert_tokens_to_ids('[No Retrieval]')

### 8.2 Decision 1: retrieve or not?

We read the model's probability for its very first token: `[Retrieval]` versus `[No Retrieval]`.

In [ ]:
# Look at the model's probability for its first token: [Retrieval] versus [No Retrieval]
def retrieve_probability(question):
    inputs = tokenizer(format_prompt(question), return_tensors='pt').to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1]            # scores for the next token
    probs = torch.softmax(logits.float(), dim=-1)
    p_retrieval, p_no_retrieval = probs[retrieval_id].item(), probs[no_retrieval_id].item()
    return p_retrieval / (p_retrieval + p_no_retrieval)

### 8.3 Generate an answer and read its critique

`generate` writes the answer with the reflection tokens kept in it. `read_critique` pulls out whether the chunk is relevant, how well the answer is supported (0 to 2), how useful it is (1 to 5), and the clean answer.

In [ ]:
def generate(prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    # Keep the reflection tokens in the text
    return tokenizer.decode(output_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=False)

support_points = {'[Fully supported]': 2, '[Partially supported]': 1, '[No support / Contradictory]': 0}

def read_critique(output):
    relevant = '[Relevant]' in output                                    # Decision 2: is the chunk relevant?
    support = next((p for t, p in support_points.items() if t in output), 0)   # Decision 3: is the answer supported?
    match = re.search(r'\[Utility:(\d)\]', output)
    utility = int(match.group(1)) if match else 0                        # Decision 4: is the answer useful (1-5)?
    answer = re.sub(r'\[[^\]]*\]|</s>', '', output).strip()              # the answer without the reflection tokens
    return {'relevant': relevant, 'support': support, 'utility': utility, 'answer': answer}

### 8.4 Decision 2 as a probability: relevant or irrelevant?

Right after a retrieved chunk, we compare the probability of `[Relevant]` versus `[Irrelevant]`.

In [ ]:
model.generation_config.max_length = None  # silences the max_length warning

relevant_id = tokenizer.convert_tokens_to_ids('[Relevant]')
irrelevant_id = tokenizer.convert_tokens_to_ids('[Irrelevant]')

# Decision 2 as a probability: [Relevant] versus [Irrelevant] right after the retrieved chunk
def relevance_probability(question, chunk):
    inputs = tokenizer(format_prompt(question, chunk), return_tensors='pt').to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1]
    probs = torch.softmax(logits.float(), dim=-1)
    p_relevant, p_irrelevant = probs[relevant_id].item(), probs[irrelevant_id].item()
    return p_relevant / (p_relevant + p_irrelevant)

## 9. The full Self-RAG loop

One function that runs all the decisions:
1. Decide whether to retrieve. Below the threshold (0.2), answer directly.
2. Otherwise, let the model critique every retrieved chunk and score each one.
3. Answer with the best chunk, and **warn** if the best chunk has no support or low relevance. Then the answer comes from the model's own knowledge, not our text.

The 0.2 threshold and the combined score (`relevance + support / 2 + utility / 5`) are simple choices made for this demo.

In [ ]:
def self_rag(question, k=3, threshold=0.2):
    print('QUESTION:', question)

    # Decision 1: retrieve or not?
    p = retrieve_probability(question)
    print(f'  Decision 1: probability of [Retrieval] = {p:.2f} ->', 'retrieve' if p > threshold else 'do not retrieve')
    if p <= threshold:
        print('  ANSWER (no retrieval):', read_critique(generate(format_prompt(question)))['answer'])
        return

    # Decisions 2-4: the model critiques every retrieved chunk
    candidates = []
    for doc in store.similarity_search(question, k=k):
        c = read_critique(generate(format_prompt(question, doc.page_content)))
        c['chunk'] = doc.page_content
        c['p_relevant'] = relevance_probability(question, doc.page_content)
        # Combined score: each critique signal scaled to 0-1, then added
        c['score'] = c['p_relevant'] + c['support'] / 2 + c['utility'] / 5
        candidates.append(c)
        print(f"  chunk: {c['chunk'][:60]}...")
        print(f"    p(relevant)={c['p_relevant']:.2f} | support={c['support']} (0-2) | utility={c['utility']} (0-5) | score={c['score']:.2f}")

    best = max(candidates, key=lambda c: c['score'])
    if best['support'] == 0 or best['p_relevant'] < 0.5:
        print("  WARNING: no retrieved chunk supports this answer. It comes from the model's own knowledge, not our text.")
    print('  ANSWER:', best['answer'])



## 10. Try it on five questions

### 10.1 Three questions: no retrieval needed, answer in the text, answer not in the text

In [ ]:
questions = [
    'What is 2+2?',                         # should not need retrieval
    'What are CNNs used for?',              # answer is in our text
    'Who won the FIFA World Cup in 2022?'   # not in our text
]

for question in questions:
    self_rag(question)
    print()

QUESTION: What is 2+2?
  Decision 1: probability of [Retrieval] = 0.00 -> do not retrieve
  ANSWER (no retrieval): 4

QUESTION: What are CNNs used for?
  Decision 1: probability of [Retrieval] = 1.00 -> retrieve
  chunk: Deep learning techniques, such as Convolutional Neural Netwo...
    p(relevant)=1.00 | support=1 (0-2) | utility=5 (0-5) | score=2.50
  chunk: Machine learning models analyze medical records, laboratory ...
    p(relevant)=1.00 | support=1 (0-2) | utility=5 (0-5) | score=2.50
  chunk: Artificial Intelligence (AI) is transforming healthcare by i...
    p(relevant)=1.00 | support=1 (0-2) | utility=5 (0-5) | score=2.50
  ANSWER: CNNs are used for a variety of tasks, such as image classification, object detection, and image segmentation.

QUESTION: Who won the FIFA World Cup in 2022?
  Decision 1: probability of [Retrieval] = 0.98 -> retrieve
  chunk: Deep learning techniques, such as Convolutional Neural Netwo...
    p(relevant)=0.59 | support=0 (0-2) | utility=5 (0-5) | 

### 10.2 Two more questions: an answer fully in one sentence, and a creative request

In [ ]:
self_rag('Which technique helps extract information from clinical notes?')   # fully answered by one sentence of our text
print()
self_rag('Write a short poem about autumn.')                                  # creative: no retrieval needed

QUESTION: Which technique helps extract information from clinical notes?
  Decision 1: probability of [Retrieval] = 0.90 -> retrieve
  chunk: Natural Language Processing (NLP) helps extract meaningful i...
    p(relevant)=1.00 | support=2 (0-2) | utility=5 (0-5) | score=3.00
  chunk: Machine learning models analyze medical records, laboratory ...
    p(relevant)=1.00 | support=1 (0-2) | utility=5 (0-5) | score=2.50
  chunk: Deep learning techniques, such as Convolutional Neural Netwo...
    p(relevant)=1.00 | support=1 (0-2) | utility=5 (0-5) | score=2.50
  ANSWER: Natural Language Processing (NLP) techniques help extract information from clinical notes.

QUESTION: Write a short poem about autumn.
  Decision 1: probability of [Retrieval] = 0.13 -> do not retrieve
  ANSWER (no retrieval): Leaves drift to the ground,
The air is crisp and around,
The sun sets slowly,
As the days grow shorter.


## 11. How to read the results

*These notes describe the run saved in this notebook. Another run may differ slightly.*

- **Decision 1 (retrieve or not):** "What is 2+2?" and the autumn poem got a low `[Retrieval]` probability (0.00 and 0.13), so they were answered directly. The other three questions got 0.90 to 1.00 and went through retrieval.
- **Answer in the text:** for the clinical-notes question, the matching chunk had relevance 1.00, support 2 (fully supported) and the highest score.
- **Partial support:** for the CNN question, all three chunks got the same scores. The answer mentions uses that are not in our text, and the model rated it only partially supported (1).
- **Answer not in the text:** for the 2022 World Cup question, no chunk supported the answer (support 0), so the warning appeared. Utility was still 5: utility says the answer looks useful, not that it is grounded in our text.